# Week 1 (starter): Tokenization Analysis

This is the starter notebook for the Week 1 assignment. It runs as-is on placeholder text so you can see the shape of each step; your job is to replace the placeholders with your own passages and analysis, then commit it to your repository and open a pull request.

Cells marked **TODO (you)** are where you do the work. Everything runs in Jupyter or Google Colab. No GPU, no API key, one dependency: `tiktoken`.

The five parts match the assignment: stand up your repo, run the analysis, evaluate with real figures, find one failure, and submit.

In [ ]:
# Setup. In Colab, uncomment the install line on first run.
!pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


## Part 1: Stand up your repository

Do this once, outside the notebook:

1. Create a public repo (suggested name `cosc-650`).
2. Add a `README.md` a stranger could read (what it is, how it is organized, the tools you use).
3. Add an agent context file that your AI tool reads, with project context and conventions. `AGENTS.md` is the cross-tool convention; `CLAUDE.md` and `GEMINI.md` are tool-specific variants. Use whichever your tool reads.
4. Work on a branch and open a pull request into `main`. You will do this every week.

Then commit this notebook into the repo and keep going.

## Helpers (provided)

Two small functions: count tokens for a string, and show the exact sub-token pieces a word breaks into. The demo uses a line you may recognize.

In [ ]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

'Panic'            -> 2 token(s): ['P', 'anic']
' towel'           -> 1 token(s): [' towel']
'42'               -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Your passages

**TODO (you):** replace the two placeholders with your own text. The non-English passage must be at least 100 words, with a faithful English translation. The placeholders below are short Hitchhiker's Guide lines so the notebook runs; swap in your real passages.

In [ ]:
english_text ="""
Once upon a time, a girl was playing in the woods and noticed a house far off in the distance.
That house was lived in by a family of three bears.
Knock, knock.
Even though she knocked on the door, there came no reply , so the girl quietly went in.
On the kitchen table there lay a large bowl, a medium bowl, and a small bowl .
Since soup was in every bowl, she decided to try some from each.
"The soup from the smallest bowl is the most delicious."
The girl ,who was incredibly hungry then ate up all the soup from the smallest bowl.
There were three chairs in the living room.
The large chair was too high.
The medium sized chair was not comfortable.
But the small sized chair was perfect for the girl.
While rocking in the chair,
CRACK!
"""
foreign_text ="""
むかしむかし、おんなのこ が もり で あそんで いると、とおく に いえ が みえた ので いってみました。
その いえ は、さんびき の クマ の おやこ が すむ いえ でした。トントントン。
ドア を たたいても へんじ が ない ので、おんなのこ は そっと なか に はいりました。
だいどころ の テーブル の うえ に、おおきい おわん と、ちゅうくらい の おわん と、ちいさい おわん が ありました。
どの おわん にも スープ が 入っていたので、おんなのこ は ひとくち ずつ のんでみました。
「ちいさな おわん の スープ が、いちばん おいしいわ」
おなか が すいていた おんなのこ は、ちいさい おわん の スープ を、すっかり のんで しまいました。
いま には、イス が みっつ ありました。
おおきい イス は、たかすぎます。
ちゅうくらい の イス は、すわりごこち が よく ありません。
ちいさい イス は、おんなのこ に ピッタリ です。
ゆらして あそんでいたら、 ドシン！
"""

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')
# TODO (you): one or two sentences interpreting these numbers for YOUR language pair.

English words: 144
Foreign words: 100
English   | chars  741 | GPT-4  167 | GPT-4o  167
Foreign   | chars  467 | GPT-4  423 | GPT-4o  325

Multilingual tax  GPT-4: 2.53x   GPT-4o: 1.95x


## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [ ]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')
# TODO (you): state what this means for a product serving users in your chosen language.

A 128,000-token window holds about 766 English copies and 302 foreign copies of your passage.
Per-request cost multiplier for the foreign language: 2.53x (billing is per token).


For a product serving Japanese-speaking users, this difference means that Japanese text may consume substantially more tokens than an equivalent amount of English text. It could increase inference costs if resources are tied to token usage. Products serving multiple languages should therefore account for differences in tokenization efficiency rather than assuming that the same number of characters will require a similar number of tokens across languages.

## Part 4: Bias splits and one failure

**TODO (you):** (a) pick three words where your non-English form fragments far worse than the English equivalent, and show both with `show_split`; (b) find ONE input whose token count defies intuition and explain it. A few failure candidates are demonstrated below to get you started; replace them with your own find and write the explanation plus a mitigation.

In [ ]:
print('English baselines:')
for w in ['Once upon a time', 'a girl', 'playing']:
    show_split(w)
print('Japanese baselines:')
for w in ['むかしむかし', 'おんなのこ', 'あそんで']:
    show_split(w)
print('\n(b) failure candidates to explore:')
show_split('\U0001F408')
show_split('08/30/2026')
show_split('August 30th, 2026')

English baselines:
'Once upon a time' -> 4 token(s): ['Once', ' upon', ' a', ' time']
'a girl'           -> 2 token(s): ['a', ' girl']
'playing'          -> 1 token(s): ['playing']
Japanese baselines:
'むかしむかし'           -> 8 token(s): ['�', '�', 'か', 'し', '�', '�', 'か', 'し']
'おんなのこ'            -> 5 token(s): ['お', 'ん', 'な', 'の', 'こ']
'あそんで'             -> 4 token(s): ['あ', 'そ', 'ん', 'で']

(b) failure candidates to explore:
'🐈'                -> 3 token(s): ['�', '�', '�']
'08/30/2026'       -> 6 token(s): ['08', '/', '30', '/', '202', '6']
'August 30th, 2026' -> 8 token(s): ['August', ' ', '30', 'th', ',', ' ', '202', '6']


For the cat emoji, tt seems like a single visible character to a user, but the tokenizer represents it with multiple tokens because emojis are not always encoded as a single token. This may make token usage higher than expected.

For mitigation, this product should account for potentially higher token usage from emojis when setting token budgets.

For the date, 08/30/2026, it may seem to be a condensed way to write the date. However, the tokenizer splits the numbers and "/" separators into multiple tokens since this particular date format may have a single efficient token representation.

The other date format is another way to show how humans are used to writing the date. The tokenizer, however, divided this into multiple tokens.

For both of these examples, the prodcut could normalize dates into a standardized representation before processing, such as 2026-08-30, when preserving the user's original formatting is not necessary. This can potentially reduce unnecessary token usage.

## Part 5: Submit

Before you open the pull request, check:

- The notebook runs top to bottom on **your** passages, not the placeholders.
- Your three bias splits are shown and explained.
- The failure case has a cause and a mitigation.
- The PR description has a one-paragraph result summary with your headline numbers.
- You linked one issue in your repo logging this as a research note (title, inputs, what you found).

Rubric: repo quality (15), counts from both tokenizers (20), tax computed (15), three bias splits (20), cost and context figures (15), the failure case (10), PR hygiene (5).